## Part 5: Final Experiments

In [60]:
import time
import mlflow
import pandas as pd
from IPython.display import display
from chat.client import ChatClient
from utils.tracker import setup_mlflow
from tools.definitions import TOOLS_LIST

## Task 5.1: Failure Modes

### Implementation

In [61]:
# setup_mlflow()
setup_mlflow("Task-5.1-Failure-Modes")

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-5.1-Failure-Modes at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [62]:
print("\n=== Failure 1: Wrong Tool Choice ===\n")

from tools.definitions import calculator, document_lookup, build_tool_from_function

client = ChatClient(model="databricks-gemma-3-12b", budgeting=False)

wrong_tool_query = "What is 25 plus 17?"

client.reset_conversation()
with mlflow.start_run(run_name="failure1-before-fix"):
    response_before = client.chat_loop(wrong_tool_query)
    mlflow.log_param("query", wrong_tool_query)
    mlflow.log_text(str(client.history), "history_before.txt")
    print("Before fix:", response_before)

calculator.__doc__ = (
    "Evaluates a basic arithmetic expression using only +, -, *, /. "
    "Use this ONLY for numeric math. Never use this for looking up facts or text."
)
fixed_tools_1 = [build_tool_from_function(calculator), build_tool_from_function(document_lookup)]

client.reset_conversation()
with mlflow.start_run(run_name="failure1-after-fix"):
    response_after = client.chat_loop(wrong_tool_query, tools=fixed_tools_1)
    mlflow.log_param("query", wrong_tool_query)
    mlflow.log_text(str(client.history), "history_after.txt")
    print("After fix:", response_after)


=== Failure 1: Wrong Tool Choice ===

Before fix: 25 plus 17 is 42.
🏃 View run failure1-before-fix at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127/runs/6411c2f9770c445b990dd697332aba40
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127
After fix: 25 plus 17 is 42.
🏃 View run failure1-after-fix at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127/runs/e2e4fd4489d4404cbacb2c8084319e24
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127


In [63]:
print("\n=== Failure 2: Loop Pressure ===\n")

loop_pressure_query = (
    "Keep double-checking the Quarterly Earnings figure as many times as you can before "
    "answering, verifying it repeatedly until you are absolutely certain."
)

client.reset_conversation()
with mlflow.start_run(run_name="failure2-before-fix"):
    response_before = client.chat_loop(loop_pressure_query, max_turns=5)
    mlflow.log_param("query", loop_pressure_query)
    mlflow.log_text(response_before, "response_before.txt")
    print("Before fix:", response_before)

fixed_query_2 = (
    "Look up the Quarterly Earnings figure exactly once, then immediately give your final answer."
)

client.reset_conversation()
with mlflow.start_run(run_name="failure2-after-fix"):
    response_after = client.chat_loop(fixed_query_2, max_turns=5)
    mlflow.log_param("query", fixed_query_2)
    mlflow.log_text(response_after, "response_after.txt")
    print("After fix:", response_after)


=== Failure 2: Loop Pressure ===

Before fix: I am ready to assist. What can I do for you?
🏃 View run failure2-before-fix at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127/runs/cf188c16318e45a8b0b6f0d7c03b2a2a
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127
After fix: The Quarterly Earnings totaled $3.2 million.
🏃 View run failure2-after-fix at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127/runs/38a08018d6894cae9754c666ed31245e
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127


In [64]:
print("\n=== Failure 3: Bad Arguments ===\n")

bad_args_query = "Use the calculator tool to compute 2 to the power of 8."

client.reset_conversation()
with mlflow.start_run(run_name="failure3-before-fix"):
    response_before = client.chat_loop(bad_args_query, max_turns=5)
    mlflow.log_param("query", bad_args_query)
    mlflow.log_text(str(client.history), "history_before.txt")
    print("Before fix:", response_before)

calculator.__doc__ = (
    "Evaluates a basic arithmetic expression. ONLY supports +, -, *, / and parentheses. "
    "Does NOT support exponents, roots, or any other operation. "
    "To compute an exponent like X^Y, carefully write out the number X exactly Y times, "
    "separated by '*' symbols (e.g., 3 to the power of 3 must be rewritten as '3 * 3 * 3'). "
    "Double check your count before sending the expression."
)
fixed_tools_3 = [build_tool_from_function(calculator), build_tool_from_function(document_lookup)]

client.reset_conversation()
with mlflow.start_run(run_name="failure3-after-fix"):
    response_after = client.chat_loop(bad_args_query, tools=fixed_tools_3, max_turns=5)
    mlflow.log_param("query", bad_args_query)
    mlflow.log_text(str(client.history), "history_after.txt")
    print("After fix:", response_after)


=== Failure 3: Bad Arguments ===

Before fix: 2 to the power of 8 is 256.
🏃 View run failure3-before-fix at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127/runs/7ec67690e8c5456c9f196e3176bdab79
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127
After fix: 2 to the power of 8 is 256.
🏃 View run failure3-after-fix at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127/runs/5ece14751dbe4d9c876e5e3b71504ec6
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976127


### Analysis Questions

1. For bad arguments, where should the check happen: before the tool runs, inside the tool, or in a separate validation step?
2. Can loop pressure happen in a normal user request, not just in a test you designed? Which kinds of requests are most likely to trigger it?
3. After adding fixes, did any fix create a new problem? Describe one second-order effect.

## Task 5.2: The Final Experiment

### Implementation

In [65]:
# setup_mlflow()
setup_mlflow("Task-5.2-Final-Experiment")

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-5.2-Final-Experiment at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [66]:

final_query = (
    "Find the reported net profit figure in the Honda annual report, then calculate "
    "its value after a 25% increase."
)

# Configure prompts based on the conditions
ptcf = (
    "**Persona:** You are a meticulous financial analyst assistant.\n\n"
    "**Task:** Find the exact fact requested, then perform the requested calculation using that fact.\n\n"
    "**Context:** The fact you need is not contained in this message; you must retrieve it from the document before answering.\n\n"
    "**Format:** State the retrieved fact, then show the calculation, then give the final numeric answer clearly.\n\n"
)

strategy_addition = (
    "Think step by step: first identify the exact fact needed, then perform the calculation, "
    "then state the final answer.\n\n"
)

tool_instruction = (
    "When you need information from the report, call the document_lookup tool with the "
    "directory parameter set to 'part5'.\n\n"
)


conditions = [
    {
        "name": "Baseline", 
        "prompt": final_query, 
        "tools": []
    },
    {
        "name": "Prompt_Only", 
        "prompt": ptcf + final_query, 
        "tools": []
    },
    {
        "name": "Prompt_and_Strategy", 
        "prompt": ptcf + strategy_addition + final_query, 
        "tools": []
    },
    {
        "name": "Prompt_Strategy_Tools", 
        "prompt": ptcf + strategy_addition + tool_instruction + final_query, 
        "tools": TOOLS_LIST
    }
]

final_results = []

for cond in conditions:
    print(f"\nRunning Condition: {cond['name']}")

    cond_client = ChatClient(model="databricks-gemma-3-12b", budgeting=False)
    start_time = time.time()

    if cond["tools"]:
        response = cond_client.chat_loop(cond["prompt"], max_turns=5, tools=cond["tools"])
    else:
        response = cond_client.chat(cond["prompt"], stream=False)

    latency = time.time() - start_time
    try:
        token_count = cond_client._estimate_history_tokens()
    except Exception:
        token_count = sum(cond_client.get_token_count(msg.get("content") or "") for msg in cond_client.history)

    quality_score = 3


    with mlflow.start_run(run_name=f"final-{cond['name']}"):
        mlflow.log_param("condition", cond["name"])
        mlflow.log_metric("quality_score", quality_score)
        mlflow.log_metric("latency_sec", latency)
        mlflow.log_metric("token_count", token_count)
        mlflow.log_text(response, f"{cond['name']}_response.txt")

    final_results.append({
        "Condition": cond["name"],
        "Quality (1-5)": quality_score,
        "Latency (s)": round(latency, 3),
        "Token Count": token_count,
        "Response (truncated)": response[:200]
    })
# Display a compact summary table
final_df = pd.DataFrame(final_results)
display(final_df)


Running Condition: Baseline
🏃 View run final-Baseline at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976128/runs/d38a7eac42544aa58141ec6bf626f290
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976128

Running Condition: Prompt_Only
🏃 View run final-Prompt_Only at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976128/runs/512ad66fa29b431c9b030153c142dd75
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976128

Running Condition: Prompt_and_Strategy
🏃 View run final-Prompt_and_Strategy at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976128/runs/906321dca8184d6dabad69f6a81c5c79
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3069368079976128

Running Condition: Prompt_Strategy_Tools
🏃 View run final-Prompt_Strategy_Tools at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experi

,Condition,Quality (1-5),Latency (s),Token Count,Response (truncated)
0,Baseline,3,5.590,346,"Okay, I can help with that! However, I need to..."
1,Prompt_Only,3,3.054,251,"Okay, I will perform that task.\n\n**Retrieved..."
2,Prompt_and_Strategy,3,2.474,232,"Okay, I'm ready. Please provide the Honda annu..."
3,Prompt_Strategy_Tools,3,4.778,387,"Okay, I need to find the net profit figure fro..."


### Analysis Questions
 
1. Create a table showing quality score vs. total token count for all four conditions. What is the marginal token cost of each quality improvement step (1 -> 2, 2 -> 3)?
2. How large is the quality gap between Condition 1 (the naive baseline) and Condition 3 (the best setup)? Is the engineering effort to close that gap worth it for every use case? Describe a scenario where Condition 1 is the right choice.